In [47]:
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [48]:
import threading
import subprocess

# Start Ollama service in background

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()

In [49]:
import subprocess
import time
import requests

def ensure_ollama_running():
    try:
        # tenta conectar
        requests.get("http://localhost:11434", timeout=2)
        print("Ollama já está rodando ✅")
        return
    except:
        print("Ollama não está rodando. Iniciando... 🚀")

    # inicia processo em background
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    # espera subir
    for _ in range(10):
        try:
            time.sleep(1)
            requests.get("http://localhost:11434", timeout=2)
            print("Ollama iniciado com sucesso ✅")
            return
        except:
            continue

    raise RuntimeError("Falha ao iniciar Ollama.")

In [50]:
!for i in {1..5}; do ollama pull qwen2.5:3b && break || sleep 5; done

In [51]:
### https://docs.ollama.com/capabilities/tool-calling#python

In [52]:
!uv pip install ollama

Using Python 3.12.12 environment at: /usr
Audited 1 package in 136ms


In [53]:
!uv pip install rich

Using Python 3.12.12 environment at: /usr
Audited 1 package in 82ms


In [54]:
import json
from rich import print
from ollama import chat
model = 'qwen2.5:3b'

In [55]:
import sqlite3
from datetime import datetime

DATABASE_NAME = 'banco.db'

In [56]:
def get_db_connection():
    """Establishes a connection to the SQLite database."""
    conn = sqlite3.connect(DATABASE_NAME)
    conn.row_factory = sqlite3.Row # Allows accessing columns by name
    return conn

In [57]:
def create_tables():
    """Cria todas as tabelas do sistema de tickets"""

    conn = get_db_connection()
    cursor = conn.cursor()

    # ---------------- USUARIOS ----------------
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS usuarios (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome TEXT NOT NULL,
            email TEXT NOT NULL UNIQUE,
            senha_hash TEXT NOT NULL,
            role TEXT NOT NULL CHECK(role IN ('admin','suporte','cliente')),
            ativo INTEGER NOT NULL DEFAULT 1,
            criado_em TEXT DEFAULT CURRENT_TIMESTAMP
        )
    """)

    # ---------------- CATEGORIAS ----------------
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS categorias (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome TEXT NOT NULL UNIQUE,
            descricao TEXT
        )
    """)

    # ---------------- TICKETS ----------------
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS tickets (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            titulo TEXT NOT NULL,
            descricao TEXT NOT NULL,
            status TEXT NOT NULL CHECK(status IN ('aberto','em_andamento','resolvido','fechado')) DEFAULT 'aberto',
            prioridade TEXT NOT NULL CHECK(prioridade IN ('baixa','media','alta','critica')) DEFAULT 'media',
            usuario_id INTEGER NOT NULL,
            responsavel_id INTEGER,
            categoria_id INTEGER,
            criado_em TEXT DEFAULT CURRENT_TIMESTAMP,
            atualizado_em TEXT,
            fechado_em TEXT,
            FOREIGN KEY(usuario_id) REFERENCES usuarios(id),
            FOREIGN KEY(responsavel_id) REFERENCES usuarios(id),
            FOREIGN KEY(categoria_id) REFERENCES categorias(id)
        )
    """)

    # ---------------- COMENTARIOS ----------------
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS comentarios (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            ticket_id INTEGER NOT NULL,
            usuario_id INTEGER NOT NULL,
            mensagem TEXT NOT NULL,
            criado_em TEXT DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY(ticket_id) REFERENCES tickets(id),
            FOREIGN KEY(usuario_id) REFERENCES usuarios(id)
        )
    """)

    # ---------------- HISTORICO STATUS ----------------
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS historico_status (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            ticket_id INTEGER NOT NULL,
            status_anterior TEXT,
            status_novo TEXT,
            alterado_por INTEGER,
            alterado_em TEXT DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY(ticket_id) REFERENCES tickets(id),
            FOREIGN KEY(alterado_por) REFERENCES usuarios(id)
        )
    """)

    # ---------------- ANEXOS ----------------
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS anexos (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            ticket_id INTEGER NOT NULL,
            nome_arquivo TEXT NOT NULL,
            caminho TEXT NOT NULL,
            enviado_por INTEGER,
            criado_em TEXT DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY(ticket_id) REFERENCES tickets(id),
            FOREIGN KEY(enviado_por) REFERENCES usuarios(id)
        )
    """)

    # ---------------- SLA POLITICAS ----------------
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS sla_politicas (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            prioridade TEXT NOT NULL UNIQUE,
            tempo_limite_horas INTEGER NOT NULL
        )
    """)

    # ---------------- LOGS SISTEMA ----------------
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS logs_sistema (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            tipo_evento TEXT NOT NULL,
            mensagem TEXT NOT NULL,
            nivel TEXT CHECK(nivel IN ('info','warning','error')) DEFAULT 'info',
            criado_em TEXT DEFAULT CURRENT_TIMESTAMP
        )
    """)

    conn.commit()
    conn.close()

    print("Banco de dados do sistema de tickets criado/atualizado com sucesso")

In [58]:
create_tables()

Banco de dados do sistema de tickets criado/atualizado com sucesso

In [59]:
def seed_data():
    conn = get_db_connection()
    cursor = conn.cursor()

    # ---------- USUARIOS ----------
    usuarios = [
        ("Admin Master", "admin@empresa.com", "hash_admin", "admin"),
        ("Suporte João", "suporte1@empresa.com", "hash_suporte1", "suporte"),
        ("Suporte Maria", "suporte2@empresa.com", "hash_suporte2", "suporte"),
        ("Cliente Pedro", "pedro@cliente.com", "hash_cliente1", "cliente"),
        ("Cliente Ana", "ana@cliente.com", "hash_cliente2", "cliente"),
    ]

    cursor.executemany("""
        INSERT OR IGNORE INTO usuarios (nome, email, senha_hash, role)
        VALUES (?, ?, ?, ?)
    """, usuarios)

    # ---------- CATEGORIAS ----------
    categorias = [
        ("Bug", "Problemas no sistema"),
        ("Infraestrutura", "Servidor, rede, deploy"),
        ("Dúvida", "Perguntas gerais"),
        ("Feature", "Solicitação de melhoria"),
        ("Financeiro", "Cobrança, pagamento")
    ]

    cursor.executemany("""
        INSERT OR IGNORE INTO categorias (nome, descricao)
        VALUES (?, ?)
    """, categorias)

    # ---------- SLA ----------
    sla = [
        ("baixa", 72),
        ("media", 48),
        ("alta", 24),
        ("critica", 4)
    ]

    cursor.executemany("""
        INSERT OR IGNORE INTO sla_politicas (prioridade, tempo_limite_horas)
        VALUES (?, ?)
    """, sla)

    # ---------- TICKETS ----------
    tickets = [
        ("Erro ao logar", "Usuário não consegue acessar o sistema.", "aberto", "alta", 4, 2, 1),
        ("Servidor fora do ar", "Site indisponível desde manhã.", "em_andamento", "critica", 5, 3, 2),
        ("Dúvida sobre cobrança", "Valor da fatura está incorreto.", "resolvido", "media", 4, 2, 5),
        ("Sugestão de melhoria", "Adicionar modo escuro.", "aberto", "baixa", 5, None, 4),
    ]

    cursor.executemany("""
        INSERT INTO tickets (titulo, descricao, status, prioridade, usuario_id, responsavel_id, categoria_id)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, tickets)

    # ---------- COMENTARIOS ----------
    comentarios = [
        (1, 2, "Estamos verificando o problema."),
        (2, 3, "Equipe de infra acionada."),
        (3, 2, "Problema resolvido, favor confirmar."),
        (1, 4, "Ainda aguardando retorno."),
    ]

    cursor.executemany("""
        INSERT INTO comentarios (ticket_id, usuario_id, mensagem)
        VALUES (?, ?, ?)
    """, comentarios)

    # ---------- HISTORICO STATUS ----------
    historico = [
        (2, "aberto", "em_andamento", 3),
        (3, "aberto", "resolvido", 2),
    ]

    cursor.executemany("""
        INSERT INTO historico_status (ticket_id, status_anterior, status_novo, alterado_por)
        VALUES (?, ?, ?, ?)
    """, historico)

    # ---------- LOGS ----------
    logs = [
        ("ticket_criado", "Ticket #1 criado", "info"),
        ("ticket_atualizado", "Ticket #2 mudou para em_andamento", "info"),
        ("erro_sistema", "Timeout na API externa", "warning"),
    ]

    cursor.executemany("""
        INSERT INTO logs_sistema (tipo_evento, mensagem, nivel)
        VALUES (?, ?, ?)
    """, logs)

    conn.commit()
    conn.close()

    print("Seed de dados inserido com sucesso")

In [60]:
seed_data()

Seed de dados inserido com sucesso

A seguir, a sessão de **Ferramentas** para o agente de dados (TOOLS)

In [61]:
import requests
import socket
import time
import datetime

SITE_URL = "https://github.com/Templasan"
HOST = "github.com"


def executar_diagnostico():
    resultado = {}
    resultado["infraestrutura"] = {}
    resultado["sistema_tickets"] = {}

    # ==============================
    # INFRAESTRUTURA
    # ==============================

    # DNS
    try:
        ip = socket.gethostbyname(HOST)
        resultado["infraestrutura"]["dns"] = {"status": "ok", "ip": ip}
    except Exception as e:
        resultado["infraestrutura"]["dns"] = {"status": "erro", "erro": str(e)}

    # Porta 443
    try:
        sock = socket.create_connection((HOST, 443), timeout=5)
        sock.close()
        resultado["infraestrutura"]["porta_443"] = "aberta"
    except Exception as e:
        resultado["infraestrutura"]["porta_443"] = f"fechada ({e})"

    # HTTP
    try:
        inicio = time.time()
        r = requests.get(SITE_URL, timeout=5)
        latencia = (time.time() - inicio) * 1000

        resultado["infraestrutura"]["http"] = {
            "status_code": r.status_code,
            "latencia_ms": round(latencia, 2),
            "online": True if r.status_code == 200 else False
        }

    except Exception as e:
        resultado["infraestrutura"]["http"] = {
            "online": False,
            "erro": str(e)
        }

    # ==============================
    # SISTEMA DE TICKETS
    # ==============================

    conn = get_db_connection()
    cursor = conn.cursor()

    # Tickets críticos abertos
    cursor.execute("""
        SELECT COUNT(*)
        FROM tickets
        WHERE prioridade = 'critica'
        AND status IN ('aberto','em_andamento')
    """)
    criticos = cursor.fetchone()[0]

    # Tickets sem responsável
    cursor.execute("""
        SELECT COUNT(*)
        FROM tickets
        WHERE responsavel_id IS NULL
        AND status IN ('aberto','em_andamento')
    """)
    sem_responsavel = cursor.fetchone()[0]

    # Tickets com SLA estourado
    cursor.execute("""
        SELECT t.id, t.criado_em, s.tempo_limite_horas
        FROM tickets t
        JOIN sla_politicas s ON t.prioridade = s.prioridade
        WHERE t.status IN ('aberto','em_andamento')
    """)

    agora = datetime.datetime.now()
    estourados = 0

    for row in cursor.fetchall():
        criado_em = datetime.datetime.fromisoformat(row[1])
        limite = row[2]
        horas_passadas = (agora - criado_em).total_seconds() / 3600

        if horas_passadas > limite:
            estourados += 1

    conn.close()

    resultado["sistema_tickets"] = {
        "tickets_criticos_abertos": criticos,
        "tickets_sem_responsavel": sem_responsavel,
        "tickets_sla_estourado": estourados
    }

    # ==============================
    # STATUS GERAL
    # ==============================

    problemas = (
        criticos > 0
        or sem_responsavel > 0
        or estourados > 0
        or not resultado["infraestrutura"]["http"].get("online", False)
    )

    resultado["status_geral"] = "ATENCAO 🚨" if problemas else "OPERACIONAL ✅"

    return resultado

In [62]:
def listar_tickets_abertos():
    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute("""
        SELECT id, titulo, prioridade, status
        FROM tickets
        WHERE status IN ('aberto','em_andamento')
        ORDER BY prioridade DESC
    """)

    rows = cursor.fetchall()
    conn.close()

    return [
        {
            "id": r[0],
            "titulo": r[1],
            "prioridade": r[2],
            "status": r[3]
        }
        for r in rows
    ]

In [63]:
def listar_tickets_criticos():
    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute("""
        SELECT id, titulo, status
        FROM tickets
        WHERE prioridade = 'critica'
        AND status IN ('aberto','em_andamento')
    """)

    rows = cursor.fetchall()
    conn.close()

    return [
        {
            "id": r[0],
            "titulo": r[1],
            "status": r[2]
        }
        for r in rows
    ]

In [64]:
def listar_tickets_sem_responsavel():
    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute("""
        SELECT id, titulo, prioridade
        FROM tickets
        WHERE responsavel_id IS NULL
        AND status IN ('aberto','em_andamento')
    """)

    rows = cursor.fetchall()
    conn.close()

    return [
        {
            "id": r[0],
            "titulo": r[1],
            "prioridade": r[2]
        }
        for r in rows
    ]

In [65]:
import datetime

def calcular_sla_restante(ticket_id: int):
    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute("""
        SELECT t.prioridade, t.criado_em, s.tempo_limite_horas
        FROM tickets t
        JOIN sla_politicas s ON t.prioridade = s.prioridade
        WHERE t.id = ?
    """, (ticket_id,))

    row = cursor.fetchone()
    conn.close()

    if not row:
        return {"erro": "Ticket não encontrado"}

    prioridade, criado_em, limite_horas = row

    criado_dt = datetime.datetime.fromisoformat(criado_em)
    agora = datetime.datetime.now()

    horas_passadas = (agora - criado_dt).total_seconds() / 3600
    horas_restantes = limite_horas - horas_passadas

    return {
        "ticket_id": ticket_id,
        "prioridade": prioridade,
        "sla_horas": limite_horas,
        "horas_restantes": round(horas_restantes, 2),
        "status_sla": "estourado 🚨" if horas_restantes < 0 else "no prazo ✅"
    }

In [66]:
def resumir_funcionalidades():
    return {
        "infraestrutura": [
            "Executar diagnóstico do site (DNS, porta 443, HTTP, latência)"
        ],
        "tickets": [
            "Listar tickets abertos ou em andamento",
            "Listar tickets críticos",
            "Listar tickets sem responsável",
            "Calcular SLA restante de um ticket"
        ],
        "monitoramento": [
            "Verificar tickets críticos ativos",
            "Identificar tickets com SLA estourado",
            "Detectar tickets sem responsável"
        ]
    }

In [67]:
SYSTEM_PROMPT = """
Você é um assistente que SEMPRE deve usar uma ferramenta
quando houver uma apropriada.

Nunca responda em texto se existir uma ferramenta compatível.
Escolha a ferramenta mais específica.
"""

MODEL_OPTIONS = {
    "temperature": 0,
    "top_p": 1,
    "num_predict": 64,
    "seed": 42,
    "keep_alive": 600
}

In [68]:
def run_agent(user_input, model, tools):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_input}
    ]

    response = chat(
        model=model,
        messages=messages,
        tools=tools,
        options=MODEL_OPTIONS
    )

    return response

In [69]:
import json

TOOL_REGISTRY = {
    # Tickets
    "listar_tickets_abertos": listar_tickets_abertos,
    "listar_tickets_criticos": listar_tickets_criticos,
    "listar_tickets_sem_responsavel": listar_tickets_sem_responsavel,
    "calcular_sla_restante": calcular_sla_restante,

    # Diagnóstico
    "executar_diagnostico": executar_diagnostico,

    # Informações
    "resumir_funcionalidades": resumir_funcionalidades
}

In [70]:
def execute_tool(tool_call):
    tool_name = tool_call.function.name

    if tool_name not in TOOL_REGISTRY:
        raise Exception(f"Tool '{tool_name}' não registrada")

    raw_args = tool_call.function.arguments

    # Se não houver argumentos
    if not raw_args:
        return TOOL_REGISTRY[tool_name]()

    # Se vier como string JSON
    if isinstance(raw_args, str):
        args = json.loads(raw_args)
    else:
        args = raw_args

    return TOOL_REGISTRY[tool_name](**args)

In [71]:
from IPython.display import clear_output

while True:
    try:
        user_input = input("Jarvis> ")

        if user_input.lower() in ["exit", "close"]:
            break

        ensure_ollama_running()

        clear_output(wait=True)
        print("Jarvis Terminal\n")

        response = run_agent(
            user_input=user_input,
            model=model,
            tools=[listar_tickets_abertos, listar_tickets_criticos, listar_tickets_sem_responsavel, calcular_sla_restante, executar_diagnostico, resumir_funcionalidades]
        )

        if response.message.tool_calls:
            tool = response.message.tool_calls[0]

            print(f"Tool chamada: {tool.function.name}")
            result = execute_tool(tool)

            print("\nResultado:")
            print(result)

        else:
            print(response.message.content)

    except KeyboardInterrupt:
        print("\nEncerrado manualmente.")
        break

Jarvis Terminal

Tool chamada: calcular_sla_restante

Resultado:

{'ticket_id': 2, 'prioridade': 'critica', 'sla_horas': 4, 'horas_restantes': 3.94, 'status_sla': 'no prazo ✅'}

Encerrado manualmente.